In [2]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx
import jax
import jax.numpy as jnp
from flax import nnx
import jax.tree_util as jtu
import sys
sys.path.insert(0, '.')


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# H2分子几何构型
bond_length = 1.4  # 平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

print("="*60)
print("H2分子系统设置")
print("="*60)
print(f"键长: {bond_length} 埃")

# 创建分子对象
mol = gto.M(atom=geometry, basis='STO-3G')

# Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"\nHartree-Fock能量: {E_hf:.8f} Ha")

# FCI计算（参考值）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print(f"\nFCI能级（参考值）:")
print("-"*50)
for i, e in enumerate(E_fcis):
    exc_energy = (e - E_fcis[0]) * 27.2114
    if i == 0:
        print(f"  E{i} (基态)     = {e:.8f} Ha")
    else:
        print(f"  E{i} (第{i}激发态) = {e:.8f} Ha  激发能: {exc_energy:.4f} eV")

# 创建NetKet哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

H2分子系统设置
键长: 1.4 埃

Hartree-Fock能量: -0.94148065 Ha

FCI能级（参考值）:
--------------------------------------------------
  E0 (基态)     = -1.01546825 Ha
  E1 (第1激发态) = -0.87542794 Ha  激发能: 3.8107 eV
  E2 (第2激发态) = -0.42938376 Ha  激发能: 15.9482 eV
  E3 (第3激发态) = -0.26922131 Ha  激发能: 20.3064 eV


In [ ]:
from NES_VMC import NES_VMC

In [5]:
# 2. 把它转为稠密矩阵 [N, N]
H_mat = ha.to_dense()  # shape: [n_spin_orbitals, n_spin_orbitals]

# 3. ✅【核心】生成 K 个直和：H ⊕ H ⊕ ... ⊕ H
K = 2  # NES-VMC 同时计算 K 个态
H_block_diag = jnp.kron(jnp.eye(K), H_mat)  # ✅ 直和！

In [ ]:
from scipy.linalg import eigh

In [ ]:
H_mat
eigenvalues, eigenvectors = eigh(H_mat)

In [ ]:
eigenvalues
eigenvectors

In [6]:
# 创建Hilbert空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

print("="*60)
print("Hilbert空间信息")
print("="*60)
print(f"空间轨道数: 2")
print(f"自旋轨道数: 4")
print(f"电子数: 2 (α=1, β=1)")
print(f"Hilbert空间维度: {hi.n_states}")
print(f"\n所有可能的电子组态:")
print(hi.all_states())

# 创建采样器
g = nk.graph.Graph(edges=[(0,1),(2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=4, spin_symmetric=True, sweep_size=64
)

Hilbert空间信息
空间轨道数: 2
自旋轨道数: 4
电子数: 2 (α=1, β=1)
Hilbert空间维度: 4

所有可能的电子组态:
[[0 1 0 1]
 [0 1 1 0]
 [1 0 0 1]
 [1 0 1 0]]


In [21]:
type(ha.to_dense())


jaxlib.xla_extension.ArrayImpl

In [8]:
import jax.tree_util as jtu

# 4.1 单态 FFNN 模型（复数值输出，匹配论文架构）
class FFNN(nnx.Module):
    def __init__(self, n_orbitals: int, hidden_dim: int = 8, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_orbitals = n_orbitals
        self.hidden_dim = hidden_dim
        # 三层全连接，复数值参数
        self.linear1 = nnx.Linear(n_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output_layer = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x: jax.Array) -> jax.Array:
        h = self.linear1(x.astype(complex))  # 输入转为复数值
        h = nnx.tanh(h)
        h = self.linear2(h)
        h = nnx.tanh(h)
        out = self.output_layer(h)
        return jnp.squeeze(out, axis=-1)  # 输出标量复数

# 4.2 初始化 4 个单态 Ansatz（K=4，对应 FCI 的 4 个态）
rng_seeds = [30, 31]  # 论文指定随机种子
n_spin_orbitals = hi.size  # 4 个自旋轨道
hidden_dim = hi.n_orbitals * 3  # 隐藏层维度=2×3=6（论文配置）

ffnn_list = [
    FFNN(n_orbitals=n_spin_orbitals, hidden_dim=hidden_dim, rngs=nnx.Rngs(seed))
    for seed in rng_seeds
]

# 验证单态输出（复数值）
test_state = hi.all_states()[0]
print(f"\n单态 Ansatz 测试输出: {ffnn_list[0](test_state)}")  # 预期类似 0.508+0.739j



单态 Ansatz 测试输出: (0.5081542321554642+0.7394651248675597j)


In [ ]:
model = FFNN(n_orbitals=4,hidden_dim=4*3,rngs=nnx.Rngs(12))
# 2. 提取参数（旧版NNX专用）
psi_matrix = jnp.array([model(i) for i in hi.all_states()])


In [25]:
ha.to_dense()@psi_matrix

Array([-0.11216814+0.02436797j, -0.35838887-0.16503862j,
        0.33530053+0.09724481j,  0.32971726-0.35658813j],      dtype=complex128)

In [26]:
model = FFNN(n_orbitals=4,hidden_dim=4*3,rngs=nnx.Rngs(12))
# 2. 提取参数（旧版NNX专用）
ffnn_state= nnx.split(model)
def forward(state,x):
  y, ffnn_state = nnx.call(state)(x)
  return y

In [ ]:
model(np.array([1,0,1,0]))

In [27]:
model_matrix = jnp.array([model(each_state) for each_state in hi.all_states()])
ha.to_dense()@model_matrix

Array([-0.11216814+0.02436797j, -0.35838887-0.16503862j,
        0.33530053+0.09724481j,  0.32971726-0.35658813j],      dtype=complex128)

In [28]:
# 创建采样器
g = nk.graph.Graph(edges=[(0,1),(2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=4, spin_symmetric=True, sweep_size=64
)
parameters = nnx.split(model)
sampler_state = sampler.init_state(forward, parameters, seed=1)
sampler_state = sampler.reset(forward, parameters, sampler_state)
# Generate samples
samples, sampler_state = sampler.sample(
    forward, parameters, state=sampler_state, chain_length=100
)
print(f"Sample shape: {samples.shape}")

Sample shape: (4, 100, 4)


In [31]:
vstate = nk.vqs.MCState(sampler, model, n_discard_per_chain=50, n_samples=1000)

In [37]:
vstate.sample(n_samples=1000,n_discard_per_chain=50).shape

(4, 250, 4)

MetropolisSamplerState(# accepted = 45999/76800 (59.89453125000001%), rng state=[4130695329  358929084])

In [ ]:
def make_netket_model(n_orbitals, hidden_dim, seed=0):
    # 1. 初始化 NNX 模型
    model = FFNN(
        n_orbitals=n_orbitals,
        hidden_dim=hidden_dim,
        rngs=nnx.Rngs(seed)
    )
    
    # 2. 提取初始参数（NNX 通用，所有版本都支持）
    initial_params = nnx.state(model)
    
    # 3. NetKet 要求的纯函数：(params, σ) → 复数值
    def netket_apply(params, σ):
        # 把参数加载回模型
        model.update(params)
        return model(σ)
    
    return netket_apply, initial_params

In [ ]:
# 创建模型 & 初始参数
log_psi_fn, initial_params = make_netket_model(
    n_orbitals=n_spin_orbitals,
    hidden_dim=6,
    seed=30
)

In [ ]:
# ✅ 完美对接 sampler.init_state
sampler = nk.sampler.MetropolisFermionHop(hi)
sampler_state = sampler.init_state(
    machine=log_psi_fn,
    parameters=initial_params,
    seed=0
)

print("✅ 全部成功！sampler 与 NNX 模型对接完成")
print("\n初始参数：", initial_params)

In [ ]:
sampler1= nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)
sampler2= nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)
FFNN_1= FFNN(n_orbitals=4,hidden_dim=4*3,rngs=nnx.Rngs(0))
FFNN_2= FFNN(n_orbitals=4,hidden_dim=4*3,rngs=nnx.Rngs(1))

vstate1 = nk.vqs.MCState(sampler1, FFNN_1, n_discard_per_chain=10, n_samples=512)
vstate2 = nk.vqs.MCState(sampler2, FFNN_2, n_discard_per_chain=10, n_samples=512)


In [38]:
class SingleStateAnsatz(nnx.Module):
    """单态 Ansatz：适配费米子系统的复数值 FFNN"""
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_spin_orbitals = n_spin_orbitals
        # 复数值线性层（费米子波函数需相位描述）
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x: jax.Array) -> jax.Array:
        """
        输入：x = 单组自旋轨道占据态（形状 [n_spin_orbitals,]，如 H₂ 的 [0,1,0,1]）
        输出：复数值波函数值 ψ(x)
        """
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)  # 输出形状 []（标量复数）

In [39]:
s = SingleStateAnsatz(n_spin_orbitals=4,hidden_dim=4*3,rngs=nnx.Rngs(1))
s([1,0,1,0])

Array(-0.20085351-1.00943396j, dtype=complex128)

In [ ]:
n_states = 2 #能态数量
single_ansatz_list = [ SingleStateAnsatz(4,4*3,rngs=nnx.Rngs(i)) for i in range(n_states)]
M = []
x_batch = [
    [0,1,0,1],  # x¹：第1组完整状态
    [1,0,1,0]   # x²：第2组完整状态
]
single_ansatz_list[0](x_batch[0])




for i in range(n_states):
    for j in range(n_states):
        # 对第 i 个单态 Ansatz，批量计算 K 组状态的输出（形状 [n_states,]）
        psi_i_xj = single_ansatz_list[j](x_batch[i])
        print(f'第{i}行 第{j}列的元素={psi_i_xj}')
        M.append(psi_i_xj)
M = jnp.stack(M, axis=0)  # 转为 JAX 数组，形状 [n_states, n_states]（K×K）
M

In [40]:
class NESTotalAnsatz(nnx.Module):
    """NES-VMC 总 Ansatz：K 个单态 Ansatz 的行列式"""
    def __init__(self, n_spin_orbitals: int, n_states: int = 3, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_states = n_states  # K
        self.n_spin_orbitals = n_spin_orbitals
        
        # 初始化 K 个独立的单态 Ansatz
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(42+i))
            for i in range(n_states)
        ]

    def __call__(self, x_batch: jax.Array) -> tuple[jax.Array, jax.Array]:
        """
        【修正版】
        输入：x_batch.shape = [K, n_spin_orbitals]
        输出：
            psi_total : 行列式标量（给 NetKet 采样）
            M_matrix  : K×K 矩阵（给 Eq.17 局部能量计算）
        """
        K = self.n_states
        M = []
        
        # 你的原版逻辑：完全不变！
        for i in range(K):
            for j in range(K):
                # M[i,j] = ψ_j(x_i)
                psi_i_xj = self.single_ansatz_list[j](x_batch[i])
                M.append(psi_i_xj)
        
        # 构建 K×K 矩阵
        M = jnp.stack(M, axis=0)
        M_matrix = M.reshape(K, K)  # 这就是论文里的 Ψ 矩阵！
        
        # 计算行列式
        psi_total = jnp.linalg.det(M_matrix)
        
        # ✅ 【关键修改】同时返回：行列式 + 矩阵
        return psi_total, M_matrix

In [43]:
# 测试环境：H₂ 系统（2 个轨道→4 个自旋轨道，K=2 个激发态）
n_spin_orbitals = 4
n_states = 2  # K=2
rngs = nnx.Rngs(seed=42)

# 初始化 Total Ansatz
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=n_spin_orbitals,
    n_states=n_states,
    hidden_dim=8,
    rngs=rngs
)

# 测试1：输入两组不同的状态（x1≠x2）
x1 = jnp.array([0,1,0,1])  # H₂ 基态自旋轨道占据态
x2 = jnp.array([1,0,1,0])  # 另一个不同状态
x_batch = jnp.stack([x1, x2], axis=0)  # 形状 [2,4]（K=2 组状态）
psi_valid,psi_matrix = total_ansatz(x_batch)
print(f"两组不同状态的行列式值：{psi_valid:.4f}")  # 非零（如 0.345+0.123j）

# 测试2：输入两组相同的状态（x1=x2，模拟态坍缩）
x_batch_collapse = jnp.stack([x1, x1], axis=0)  # 形状 [2,4]（两组状态相同）
psi_collapse,matrix = total_ansatz(x_batch_collapse)
print(f"两组相同状态的行列式值：{psi_collapse:.4f}")  # 接近 0（如 1e-6+2e-6j）

两组不同状态的行列式值：2.4032+3.0742j
两组相同状态的行列式值：0.0000+0.0000j


In [48]:
vstate.sample().shape

(4, 250, 4)

In [ ]:
def Total_ansatz_wrapper(n_spin_orbitals,n_states,rngs,x:jnp.array,K:int):
    total_ansatz = NESTotalAnsatz(
        n_spin_orbitals=n_spin_orbitals,
        n_states=n_states,
        hidden_dim=n_spin_orbitals*3,
        rngs=rngs
    )
    if K is None:
        K = x.shape[0]
    elif K != x.shape[0]:
        raise ValueError("K must be equal to the batch size of x input")
    
    
    

In [ ]:
def create_nes_mcstate(hi: nk.hilbert.SpinOrbitalFermions, n_states: int = 4, hidden_dim: int = 16, n_samples: int = 512):
    """
    创建 NES-VMC 的 MCState（采样+总 Ansatz 封装）
    hi: 希尔伯特空间（如 H₂ 的 SpinOrbitalFermions）
    n_states: K（激发态数量）
    """
    # 1. 初始化总 Ansatz
    rngs = nnx.Rngs(seed=42)
    total_ansatz = NESTotalAnsatz(
        n_spin_orbitals=hi.size,  # hi.size = 自旋轨道总数（H₂ 为 4）
        n_states=n_states,
        hidden_dim=hidden_dim,
        rngs=rngs
    )
    
    # 2. 定义“拼接状态→总波函数”的映射（适配 MCState 输入要求）
    def ansatz_wrapper(x_flat: jax.Array) -> jax.Array:
        """
        输入：x_flat = 拼接后的 K 组状态（形状 [n_states * n_spin_orbitals,]）
        输出：总波函数值 Ψ（标量复数）
        """
        # 拆分拼接状态为 K 组独立状态（形状 [n_states, n_spin_orbitals]）
        x_batch = jnp.reshape(x_flat, (n_states, hi.size))
        return total_ansatz(x_batch)
    
    # 3. 构建扩展希尔伯特空间：K 个原始希尔伯特空间的直积
    # （采样时生成 K 组独立状态，拼接为单个向量输入 ansatz_wrapper）
    hi_nes = hi ** n_states  # 直积空间：形状 [n_states * n_spin_orbitals,]
    
    # 4. 费米子采样器（适配直积空间，保证自旋对称性）
    graph = nk.graph.Graph(edges=[(0,1), (2,3)])  # H₂ 自旋轨道连接图（可根据系统调整）
    sampler = nk.sampler.MetropolisFermionHop(
        hi=hi_nes,
        graph=graph,
        n_chains=16,
        spin_symmetric=True,
        sweep_size=64
    )
    
    # 5. 封装为 NETKET MCState（采样+波函数+梯度计算）
    mc_state = nk.vqs.MCState(
        sampler=sampler,
        model=ansatz_wrapper,
        n_samples=n_samples,
        n_discard_per_chain=10,  # 丢弃初始不稳定采样
        dtype=complex  # 复数值波函数
    )
    
    return mc_state, total_ansatz

In [ ]:
# 这就是 Eq.17 的 JAX 版
def expected_total_energy(model, x):
    # 1. 前向传播 → 得到矩阵 Ψ(x) [K, K]
    _, Psi = model(x)  

    # 2. 求逆
    Psi_inv = jnp.linalg.inv(Psi)  

    # 3. 计算 Ψ⁻¹ H Ψ（H 是哈密顿量矩阵）
    H_Psi = H @ Psi  
    invPsi_H_Psi = Psi_inv @ H_Psi  

    # 4. 迹 = 所有态能量和（Eq.17 核心！）
    total_energy = jnp.trace(invPsi_H_Psi)  

    return total_energy  # 标量 → 完美支持 JAX 自动微分

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx

# ==========================================
# 【核心】自动微分兼容：从 Eq.17 得到每个 E_k(x)
# ==========================================
def local_energies_from_eq17(model, x, H):
    """
    输入：
      x: [K, n_spin_orb]
      H_block: [K*N, K*N] 块对角直和哈密顿量
    输出：
      E_locs: [K] 每个态的局部能量
    """
    _, Psi = model(x)    # Psi: [K, K]
    Psi_inv = jnp.linalg.inv(Psi)
    # ✅ 正确矩阵乘法：维度完全匹配
    M = Psi_inv @ H @ Psi    # Eq.17
    # 对角元 = 各个态的局部能量
    E_locs = jnp.diag(M)
    return E_locs


## ==============================================================================
# 2. NES-VMC 损失函数（论文 Eq.21：方差和）
# ==============================================================================
def nes_vmc_loss(model, x_batch, H):
    """
    x_batch: [batch_size, K, n_spin_orbitals]
    返回：标量损失（所有态局部能量方差之和）
    """
    # 对 batch 中每一组 K 个构型计算 E_locs
    E_locs_batch = jax.vmap(lambda x: local_energies_from_eq17(model, x, H))(x_batch)  # [B, K]

    # 每个态的平均能量 E_k
    E_k = jnp.mean(E_locs_batch, axis=0)  # [K]

    # 方差损失
    loss = jnp.mean((E_locs_batch - E_k[None, :]) ** 2)  # 标量

    return loss

In [ ]:
total_ansatz(samples[:,0,:])


In [ ]:
ha.to_dense()

In [ ]:
local_energies_from_eq17(model=total_ansatz,x=samples[:,0,:],H=ha.to_dense())

In [ ]:
# ==============================================================================
# 4. NetKet 采样：一次性生成 B 组 × K 个构型（NES-VMC 标准输入）
# ==============================================================================
def sample_nes_batch(sampler, model, batch_size, K):
    """
    生成 NES-VMC 需要的输入：shape = [batch_size, K, n_spin_orbitals]
    """
    # NetKet 一次性采样 B*K 个独立构型
    samples = sampler.sample(model, n_samples=batch_size * K)  # [B*K, n_orb]
    
    #  reshape 成 NES-VMC 需要的格式
    n_orb = samples.shape[-1]
    samples = samples.reshape(batch_size, K, n_orb)  # [B, K, n_orb]
    return samples


In [ ]:
@nnx.jit
def compute_nes_gradient(model: nnx.Module, x_batch, H):
    """
    输入：
        model: 你的 NESTotalAnsatz
        x_batch: [B, K, n_orb]  → 来自 NetKet 采样器
        H: 哈密顿量矩阵
    输出：
        loss: 标量损失
        grads: 模型参数的梯度
    """
    # 自动微分：对 model 求梯度
    loss, grads = nnx.value_and_grad(nes_vmc_loss)(model, x_batch, H)
    return loss, grads


In [ ]:
# ==============================================================================
# 4. NetKet 采样：一次性生成 B 组 × K 个构型（NES-VMC 标准输入）
# ==============================================================================
def sample_nes_batch(sampler, model, batch_size, K):
    """
    生成 NES-VMC 需要的输入：shape = [batch_size, K, n_spin_orbitals]
    """
    # NetKet 一次性采样 B*K 个独立构型
    samples = sampler.sample(model, n_samples=batch_size * K)  # [B*K, n_orb]
    
    #  reshape 成 NES-VMC 需要的格式
    n_orb = samples.shape[-1]
    samples = samples.reshape(batch_size, K, n_orb)  # [B, K, n_orb]
    return samples


In [ ]:
K = 2                  # NES-VMC 同时算 K 个态
n_spin_orb = 4
batch_size = 32
hidden_dim = 4*3

# 1. 初始化你的 NESTotalAnsatz
model = NESTotalAnsatz(
    n_spin_orbitals=n_spin_orb,
    n_states=K,
    hidden_dim=hidden_dim,
    rngs=nnx.Rngs(42)
)
   # 3. NetKet 希尔伯特空间 + 采样器
# 创建Hilbert空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

In [ ]:
x_batch = sample_nes_batch(sampler, model, batch_size, K)